In [1]:
import pandas as pd
import numpy as np
import os

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    SeasonalNaive,
    Theta,
    MSTL,
    HoltWinters
)

from utilsforecast.plotting import plot_series

In [2]:
import os

print(os.getcwd())
print(os.listdir(".."))
print(os.listdir("../data"))

c:\Users\HP\Documents\timeseries_analysis\notebook
['.git', 'code', 'data', 'notebook']
['eia_region_data.csv', 'electricity_data.csv', 'prepared_data.csv']


In [3]:
# --- StatsForecast Modeling ---
# Tell Nixtla / utilsforecast that unique_id is a column (not an index)
import os
os.environ["NIXTLA_ID_AS_COL"] = "1"

In [6]:
ts = pd.read_csv("../data/prepared_data.csv", parse_dates=["ds"])
ts.head()

,unique_id,ds,y
0,1,2026-01-05 21:00:00,24
1,1,2026-01-05 21:00:00,27680
2,1,2026-01-05 21:00:00,4116
3,1,2026-01-05 21:00:00,-1216
4,1,2026-01-05 21:00:00,16224


In [8]:

# --- Train/test split: last 72 hours ---
test_length = 24 # hours
end = ts["ds"].max()
train_end = end - pd.Timedelta(hours=test_length)


train = ts[ts["ds"] <= train_end]
test  = ts[ts["ds"] > train_end]

# plot_series(train, engine="plotly")
print("Train rows:", len(train))
print("Test rows:", len(test))

# plot_series(train, engine="plotly")
# p= plot_series(test, engine="plotly")
# p.update_layout(height=400)


Train rows: 2784
Test rows: 2216


In [9]:
# --- Model Training and Forecasting ---
auto_arima = AutoARIMA(season_length=24)
s_naive = SeasonalNaive(season_length=24)
theta   = Theta(season_length=24)

mstl = MSTL(
    season_length=[24, 168],
    trend_forecaster=AutoARIMA(),
    alias="MSTL_AutoARIMA"
)

mstl2 = MSTL(
    season_length=[24, 168],
    trend_forecaster=HoltWinters(),
    alias="MSTL_HoltWinters"
)

In [10]:
# Initialize StatsForecast with models
statmodels = [auto_arima, s_naive, theta, mstl, mstl2]

# Instantiate StatsForecast
sf = StatsForecast(
    models=statmodels,
    freq="h",
    n_jobs=-1,
    fallback_model=AutoARIMA()
)

In [11]:
# Forecast
forecast_stats = sf.forecast(df=train, h=test_length, level=[95])

# Plot
p = plot_series(test, forecast_stats, engine="plotly", level=[95])
p.update_layout(height=400)
p

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\_plotly_utils\basevalidators.py:107: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result

